In [0]:
%sql
WITH tb_pedidos AS (
  SELECT *
  FROM workspace.olist.orders
  WHERE order_purchase_timestamp < '2018-07-01'
),

base AS (
  SELECT DISTINCT
      toi.seller_id,
      tor.order_id,
      tor.review_score,
      tp.order_purchase_timestamp
  FROM tb_pedidos AS tp
  LEFT JOIN olist.order_reviews AS tor
      ON tp.order_id = tor.order_id
  LEFT JOIN olist.order_items AS toi
      ON tp.order_id = toi.order_id
)

SELECT
    seller_id,

    -- Média e desvio padrão - Lifetime
    ROUND(AVG(review_score), 2) AS media_avaliacoes_lifetime,
    ROUND(STDDEV(review_score), 2) AS desvpad_avaliacoes_lifetime,

    -- Média e desvio padrão - D28
    ROUND(AVG(CASE
        WHEN DATE_DIFF('2018-07-01', order_purchase_timestamp) <= 28
        THEN review_score
    END), 2) AS media_avaliacoes_d28,

    ROUND(STDDEV(CASE
        WHEN DATE_DIFF('2018-07-01', order_purchase_timestamp) <= 28
        THEN review_score
    END), 2) AS desvpad_avaliacoes_d28,

    -- Média e desvio padrão - D56
    ROUND(AVG(CASE
        WHEN DATE_DIFF('2018-07-01', order_purchase_timestamp) <= 56
        THEN review_score
    END), 2) AS media_avaliacoes_d56,

    ROUND(STDDEV(CASE
        WHEN DATE_DIFF('2018-07-01', order_purchase_timestamp) <= 56
        THEN review_score
    END), 2) AS desvpad_avaliacoes_d56,

    -- Média e desvio padrão - D365
    ROUND(AVG(CASE
        WHEN DATE_DIFF('2018-07-01', order_purchase_timestamp) <= 365
        THEN review_score
    END), 2) AS media_avaliacoes_d365,

    ROUND(STDDEV(CASE
        WHEN DATE_DIFF('2018-07-01', order_purchase_timestamp) <= 365
        THEN review_score
    END), 2) AS desvpad_avaliacoes_d365

FROM base
WHERE seller_id IS NOT NULL
GROUP BY seller_id;

In [0]:
-- Métricas implementadas:
-- 3. Média das avaliações recebidas pelos vendedores
-- 4. Desvio padrão das avaliações
--
-- As métricas foram calculadas para os períodos:
-- D-28, D-56, D-365 e Lifetime,
-- utilizando order_purchase_timestamp como referência temporal,
-- conforme padrão definido pelo grupo.